# Phase 1: LIAR Benchmark Dataset Exploration & EDA

This notebook conducts Exploratory Data Analysis (EDA) on the **LIAR** dataset (Wang, ACL 2017).
We investigate:
1. Dataset integrity across official train, validation, and test splits
2. Six-way veracity distribution (`pants-fire`, `false`, `barely-true`, `half-true`, `mostly-true`, `true`)
3. Binary mapping (`Fake` vs. `Real`)
4. Statement length and vocabulary statistics
5. Speaker credibility and historical veracity counts

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.load_dataset import load_all_raw_splits, load_processed_split
from src.data.label_mapping import SIX_WAY_LABELS, LABEL_METADATA

print("Libraries loaded successfully!")

## 1. Load Processed Official Splits

In [ ]:
train_df = load_processed_split("train", data_dir=PROJECT_ROOT / "data" / "processed")
val_df = load_processed_split("valid", data_dir=PROJECT_ROOT / "data" / "processed")
test_df = load_processed_split("test", data_dir=PROJECT_ROOT / "data" / "processed")

print(f"Train records:      {len(train_df):,}")
print(f"Validation records: {len(val_df):,}")
print(f"Test records:       {len(test_df):,}")
print(f"Total benchmark:    {len(train_df) + len(val_df) + len(test_df):,}")

train_df.head(3)

## 2. Six-Way Label Distribution Analysis

In [ ]:
splits_summary = pd.DataFrame({
    "Train": train_df["label"].value_counts(normalize=True)[SIX_WAY_LABELS] * 100,
    "Valid": val_df["label"].value_counts(normalize=True)[SIX_WAY_LABELS] * 100,
    "Test": test_df["label"].value_counts(normalize=True)[SIX_WAY_LABELS] * 100
})

display(splits_summary.round(2))

plt.figure(figsize=(10, 5))
colors = [LABEL_METADATA[l]["color"] for l in SIX_WAY_LABELS]
train_df["label"].value_counts()[SIX_WAY_LABELS].plot(kind="bar", color=colors, edgecolor="black")
plt.title("LIAR Benchmark - 6-Way Label Distribution (Training Set)", fontsize=14, fontweight="bold")
plt.xlabel("Veracity Rating", fontsize=12)
plt.ylabel("Number of Statements", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Binary Classification Distribution (Fake vs Real)

In [ ]:
binary_counts = train_df["label_binary_name"].value_counts()
print("Binary breakdown (Train):")
print(binary_counts)
print("\nPercentages:")
print((binary_counts / len(train_df) * 100).round(2))

plt.figure(figsize=(6, 4))
binary_counts.plot(kind="bar", color=["#dc2626", "#16a34a"], edgecolor="black")
plt.title("Binary Label Distribution (Fake vs Real)", fontsize=12, fontweight="bold")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Statement Length & Word Count Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

train_df["word_count"].hist(bins=40, ax=axes[0], color="#3b82f6", edgecolor="black")
axes[0].set_title("Distribution of Statement Word Count", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")
axes[0].set_xlim(0, 100)

train_df.boxplot(column="word_count", by="label", ax=axes[1], grid=True)
axes[1].set_title("Word Count by Label", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Word Count")
axes[1].set_ylim(0, 80)
plt.suptitle("")
plt.tight_layout()
plt.show()

print(train_df[["char_length", "word_count"]].describe())

## 5. Speaker Credibility Analysis

In [ ]:
top_speakers = train_df["speaker"].value_counts().head(10)
print("Top 10 Speakers in Training Set:")
print(top_speakers)

speaker_pivot = train_df[train_df["speaker"].isin(top_speakers.index)].groupby(["speaker", "label_binary_name"]).size().unstack(fill_value=0)
speaker_pivot["Fake_Ratio"] = (speaker_pivot["Fake"] / (speaker_pivot["Fake"] + speaker_pivot["Real"]) * 100).round(1)
speaker_pivot.sort_values(by="Fake_Ratio", ascending=False)